# Real-World Example: Correcting Fornax dSph Kinematics

In this notebook, we will use the `persprot` package to correct real line-of-sight velocities for perspective rotation. We will:
1. Download a public dataset of Fornax kinematics (Walker et al. 2009) using `astroquery`.
2. Filter the catalog for highly probable member stars.
3. Apply the `persprot` correction.
4. Visualize the apparent velocity gradient before and after the correction.

In [1]:
# Install prerequisites if you don't have them
# !pip install astroquery astropy matplotlib numpy git+https://github.com/YourUsername/persprot.git

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astroquery.vizier import Vizier
import persprot as pr

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

ModuleNotFoundError: No module named 'persprot'

### 1. Fetching Public Data from VizieR
We will query the VizieR catalog **J/AJ/137/3100** (Walker+, 2009), which contains thousands of stellar radial velocities for Milky Way dwarf spheroidals. We will extract the Right Ascension, Declination, Heliocentric Radial Velocity (HRV), and membership probability for Fornax.

In [ ]:
print("Querying VizieR for Walker et al. 2009 kinematics...")

# Configure Vizier to fetch all rows and specific columns
v = Vizier(columns=['Target', 'RAJ2000', 'DEJ2000', 'HRV', 'e_HRV', 'PMem'])
v.ROW_LIMIT = -1

# Fetch the catalog
catalogs = v.get_catalogs('J/AJ/137/3100/stars')
data = catalogs[0]

# Filter for Fornax stars with a >95% probability of membership
# Note: VizieR sometimes returns strings as byte arrays (e.g., b'Fornax')
is_fornax = np.array(['Fornax' in str(target) for target in data['Target']])
is_member = data['PMem'] > 0.95
mask = is_fornax & is_member

fornax_data = data[mask]
print(f"Found {len(fornax_data)} probable member stars in Fornax.")

# Extract arrays and attach Astropy units
ra = fornax_data['RAJ2000'] * u.deg
dec = fornax_data['DEJ2000'] * u.deg
v_los = fornax_data['HRV'] * (u.km / u.s)
e_v_los = fornax_data['e_HRV'] * (u.km / u.s)

Querying VizieR for Walker et al. 2009 kinematics...


NameError: name 'Vizier' is not defined

### 2. Applying Perspective Rotation Corrections
Now we use our `persprot` package. We just need to pass the arrays of coordinates and velocities, and specify `system_name="Fornax"`. 

The package will automatically query the Local Volume Database (LVDB) to get Fornax's systemic center, proper motions, distance, and their uncertainties, and apply Equations 7, 8, and 9 to correct the data.

In [2]:
# Get Fornax center from LVDB (we can use the private _get_lvdb for a quick lookup in this notebook)
lvdb = pr.core._get_lvdb()
row = lvdb[lvdb['key_lower'] == 'fornax'][0]
ra0 = row['ra']

# Calculate relative RA offset in degrees for plotting
delta_ra = (ra.value - ra0) * np.cos(np.radians(row['dec']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Plot 1: Uncorrected
ax1.errorbar(delta_ra, v_los.value, yerr=e_v_los.value, fmt='o', 
             color='tomato', alpha=0.3, markersize=3, zorder=1)
# Add a linear trendline to highlight the gradient
z1 = np.polyfit(delta_ra, v_los.value, 1)
ax1.plot(delta_ra, np.poly1d(z1)(delta_ra), color='darkred', lw=2, label=f'Raw Gradient')
ax1.set_title("Uncorrected Line-of-Sight Velocities")
ax1.set_xlabel(r"$\Delta \alpha \cos(\delta)$ [deg]")
ax1.set_ylabel(r"$V_{los}$ [km/s]")
ax1.invert_xaxis() # Standard astronomical convention (East to left)
ax1.legend()

# Plot 2: Corrected
ax2.errorbar(delta_ra, v_pc.value, yerr=e_v_pc.value, fmt='o', 
             color='cornflowerblue', alpha=0.3, markersize=3, zorder=1)
z2 = np.polyfit(delta_ra, v_pc.value, 1)
ax2.plot(delta_ra, np.poly1d(z2)(delta_ra), color='darkblue', lw=2, label=f'Corrected Gradient')
ax2.set_title("Perspective-Corrected Velocities")
ax2.set_xlabel(r"$\Delta \alpha \cos(\delta)$ [deg]")
ax2.invert_xaxis()
ax2.legend()

plt.suptitle("Impact of Perspective Rotation on Fornax dSph Kinematics", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print(f"Original apparent slope across RA: {z1[0]:.2f} km/s/deg")
print(f"Corrected intrinsic slope across RA: {z2[0]:.2f} km/s/deg")

NameError: name 'pr' is not defined

### 3. Visualizing the Impact of Perspective Rotation
Because Fornax has a large spatial extent and significant proper motion, it exhibits a strong *apparent* velocity gradient across the sky due to perspective rotation. 

Below, we plot the line-of-sight velocities against Right Ascension (relative to the system center). You will see that the uncorrected data (red) shows a distinct slope (gradient), while the corrected data (blue) is significantly flattened, showing that much of the observed gradient was a kinematic illusion.

In [3]:
# Get Fornax center from LVDB (we can use the private _get_lvdb for a quick lookup in this notebook)
lvdb = pr.core._get_lvdb()
row = lvdb[lvdb['key_lower'] == 'fornax'][0]
ra0 = row['ra']

# Calculate relative RA offset in degrees for plotting
delta_ra = (ra.value - ra0) * np.cos(np.radians(row['dec']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Plot 1: Uncorrected
ax1.errorbar(delta_ra, v_los.value, yerr=e_v_los.value, fmt='o', 
             color='tomato', alpha=0.3, markersize=3, zorder=1)
# Add a linear trendline to highlight the gradient
z1 = np.polyfit(delta_ra, v_los.value, 1)
ax1.plot(delta_ra, np.poly1d(z1)(delta_ra), color='darkred', lw=2, label=f'Raw Gradient')
ax1.set_title("Uncorrected Line-of-Sight Velocities")
ax1.set_xlabel(r"$\Delta \alpha \cos(\delta)$ [deg]")
ax1.set_ylabel(r"$V_{los}$ [km/s]")
ax1.invert_xaxis() # Standard astronomical convention (East to left)
ax1.legend()

# Plot 2: Corrected
ax2.errorbar(delta_ra, v_pc.value, yerr=e_v_pc.value, fmt='o', 
             color='cornflowerblue', alpha=0.3, markersize=3, zorder=1)
z2 = np.polyfit(delta_ra, v_pc.value, 1)
ax2.plot(delta_ra, np.poly1d(z2)(delta_ra), color='darkblue', lw=2, label=f'Corrected Gradient')
ax2.set_title("Perspective-Corrected Velocities")
ax2.set_xlabel(r"$\Delta \alpha \cos(\delta)$ [deg]")
ax2.invert_xaxis()
ax2.legend()

plt.suptitle("Impact of Perspective Rotation on Fornax dSph Kinematics", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print(f"Original apparent slope across RA: {z1[0]:.2f} km/s/deg")
print(f"Corrected intrinsic slope across RA: {z2[0]:.2f} km/s/deg")

NameError: name 'pr' is not defined